In [26]:
import os
import pandas as pd
import re

# ----------------------------
# SETTINGS
# ----------------------------
CITATIONS_CSV = "../resolutions/ga_citations_1946_2019_OLD.csv"
RESOLUTIONS_CSV = "../resolutions/ga_resolutions_1946_2019.csv"

# ----------------------------
# LOAD DATA
# ----------------------------
df1 = pd.read_csv(CITATIONS_CSV)
df2 = pd.read_csv(RESOLUTIONS_CSV)
print(f"{CITATIONS_CSV} and {RESOLUTIONS_CSV} have been loaded.")

../resolutions/ga_citations_1946_2019_OLD.csv and ../resolutions/ga_resolutions_1946_2019.csv have been loaded.


In [27]:
# ----------------------------
# PARAMETERS
# ----------------------------
WINDOW = 100
PATTERN = re.compile(r"convention", flags=re.IGNORECASE)
pd.reset_option('display.max_colwidth')

# We take the last session (years) because I can assume they are correctly
latest_session = df2['session_reg'].max()

df_aux = df2[df2['session_reg'] == latest_session].copy()
df_aux['content'] = df_aux['content'].str.strip()

#df_aux = df_aux[df_aux['content'].str.contains("treaty")]

print(len(df_aux['res_id2'].unique()))

rows = []

for _, row in df_aux.iterrows():
    text = row["content"]
    res_id2 = row["res_id2"]

    # Find all matches
    matches = list(PATTERN.finditer(text))
    if not matches:
        continue

    # Initial windows around each match
    intervals = [
        (
            max(0, m.start() - WINDOW),
            min(len(text), m.end() + WINDOW)
        )
        for m in matches
    ]

    # Merge overlapping intervals
    merged = []
    for start, end in sorted(intervals):
        if not merged:
            merged.append([start, end])
        else:
            last_start, last_end = merged[-1]
            if start <= last_end:   # overlap (or touching)
                merged[-1][1] = max(last_end, end)
            else:
                merged.append([start, end])

    # Extract text for each merged window
    for start, end in merged:
        rows.append({
            "res_id2": res_id2,
            "window": text[start:end]
        })


# New dataframe
df_windows = pd.DataFrame(rows)

print(f"Found {len(df_windows)} windows containing the word '{PATTERN.__getattribute__('pattern')}'.")


353
Found 1150 windows containing the word 'convention'.


In [28]:
df_windows = df_windows.head(20)

prompt = """What convention or treaty appear in the text?"""

Text of resolution A/RES/73/2: ...diets and lifestyles; 22. accelerate the implementation of the world health organization frame work convention on tobacco control 6 by its states parties, while continuing to implement tobacco control measures  ... 


Text of resolution A/RES/73/2: ... tobacco industry interference and to encourage other countries to consider becoming parties to the convention; 23. implement cost-effective and evidence-based interventions to halt the rise of overweight and o ... 


Text of resolution A/RES/73/2: ...l strategy to reduce the harmful use of alcohol, as well as the world health organization framework convention on tobacco control. 6 united nations, treaty series, vol. 2302, no. 41032. 4/7 18-16893 political d ... 


Text of resolution A/RES/73/15: ...the european court of human rights in ensuring effective human rights protection under the european convention for the protection of human rights and fundamental freedoms for the more than 800 million persons 

"\nmaybe also note that when 1 resolution has more than one mention, state the other of this mention. The first mention may include the title, whereas the second may just say: 'the treaty'\n"